# Compare bursting ratio

This notebook analyzes the relationship of RA-AP to the bursts.

## Initialization - don't change

In [24]:
import sys
sys.path.append('..')

from pathlib import Path

import pandas as pd

from workflows.burst_analysis import (
    analyze_burst_probabilities,
    analyze_spike_neighborhood,
    plot_prob_change_per_cell,
    plot_prob_change_per_group,
    plot_spiketype_prob_diff_per_group,
    plot_neighborhood_count_per_group,
    plot_box_and_strip_per_group
)

from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

data_params = InputDataParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [25]:
palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Input-output settings

Base directory paths
 - `MAT_DIR` - base dir of the raw **mat** files, grouped into subfolders
 - `PROCESSED_DIR` - base dir of the preprocessed data and results

In [26]:
DIR_PATH = 'C:\\tmp\\Bozso2025\\ForManuscript'

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [27]:
paths = {
    name: Path(DIR_PATH) / subdir
    for name, subdir in zip(
        ['result', 'figure', 'statistics'],
        ['01_results', '02_figures', '03_statistics']
    )
}
for path in paths.values():
    path.mkdir(parents=True, exist_ok=True)

## Analysis parameters

Name of the spike types

In [28]:
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

List of the groups in the order of display.

In [29]:
group_list = [
    'FSI_L2-3',
    'RSI_L2-3',
    'PYR_L2-3',
    'FSI_L4-6',
    'RSI_L4-6',
    'PYR_L4-6'
]

List of lists of groups to test together.

In [30]:
test_groupings = [
    ('supra', ['FSI_L2-3', 'RSI_L2-3', 'PYR_L2-3']),
    ('infra', ['FSI_L4-6', 'RSI_L4-6', 'PYR_L4-6'])
]

In [31]:
tested_feature = 'p_bursting_spikes'

## Plot parameters

## Run analysis

Collect input files according to the table.

In [32]:
results_df = pd.read_excel(f'{paths["result"]}/raap_burst_analysis.xlsx')

In [ ]:
plot_box_and_strip_per_group(
    data=results_df,
    group_names=group_list,
    palette=palette['cell_type'],
    feature_name='p_bursting_spikes',
    save_path=f'{paths["figure"]}',
    plot_title='Bursting Probability',
)

In [34]:
from itertools import combinations
from scipy.stats import kruskal, ranksums
from statsmodels.stats.multitest import multipletests

test_results = []
for grouping_name, test_groups in test_groupings:
    testing_data = results_df[results_df['group'].isin(test_groups)]
    values = [
        testing_data.loc[testing_data['group'] == group][tested_feature].to_numpy()
        for group in test_groups
    ]
    _, kruskal_p = kruskal(*values)

    p_raws = []
    pair_names = []
    for g1_i, g2_i in combinations(range(len(values)), 2):
        _, ranksums_p = ranksums(values[g1_i], values[g2_i])
        p_raws.append(ranksums_p)
        pair_names.append((test_groups[g1_i], test_groups[g2_i]))
    p_adjs = multipletests(p_raws, alpha=0.05, method='holm')[1]
    
    for i, (p_raw, p_adj) in enumerate(zip(p_raws, p_adjs)):
        test_results.append({
            'grouping': grouping_name,
            'kruskal_p': kruskal_p,
            'group_1': pair_names[i][0],
            'group_2': pair_names[i][1],
            'p_raws': p_raw,
            'p_adjs': p_adj
        })
test_results_df = pd.DataFrame(test_results)
test_results_df.to_excel(
    f'{paths["statistics"]}/bursting_between_groups.xlsx',
    index=False
)

In [35]:
lower_raap_cells = results_df[
    results_df['p_raap_if_burst'] < results_df['p_aisap_if_burst']
]
higher_raap_cells = results_df[
    results_df['p_raap_if_burst'] > results_df['p_aisap_if_burst']
]

In [ ]:
plot_spiketype_prob_diff_per_group(
    data=lower_raap_cells,
    group_names=group_list,
    palette=palette,
    save_path=f'{paths["figure"]}',
    suffix='_lower_raap'
)
plot_spiketype_prob_diff_per_group(
    data=higher_raap_cells,
    group_names=group_list,
    palette=palette,
    save_path=f'{paths["figure"]}',
    suffix='_higher_raap'
)

In [45]:
from scipy.stats import  wilcoxon

burst_condition_results = []
for subgroup_name, subgroup_df in zip(
    ['lower_raap', 'higher_raap'],
    [lower_raap_cells, higher_raap_cells]
):
    for group_name in group_list:
        group_data = subgroup_df[subgroup_df['group'] == group_name].copy()
        if group_data.empty:
            burst_condition_results.append({
                'subgroup': subgroup_name,
                'group': group_name,
                'median_1': float('nan'),
                'median_2': float('nan'),
                'diff_median': float('nan'),
                'wilcoxon_p': float('nan'),
                'ranksums_p': float('nan')
            })
            continue
        group1 = group_data['p_raap_if_burst'] * 100
        group2 = group_data['p_aisap_if_burst'] * 100
        diffs = group1 - group2
        _, wilcoxon_p = wilcoxon(
            diffs,
            alternative=('less' if subgroup_name == 'lower_raap' else 'greater')
        )
        _, ranksums_p = ranksums(group1, group2)
        burst_condition_results.append({
            'subgroup': subgroup_name,
            'group': group_name,
            'median_raap': group1.median(),
            'median_aisap': group2.median(),
            'raap_minus_aisap_median': diffs.median(),
            'wilcoxon_p': wilcoxon_p,
            'ranksums_p': ranksums_p
        })
burst_condition_results_df = pd.DataFrame(burst_condition_results)
burst_condition_results_df.to_excel(
    f'{paths["statistics"]}/spiketype_in_burst_within_groups.xlsx',
    index=False
)